# MarketPulse AI - Demand Forecasting & Market Signals
### End-to-End Key Results Reproduction Notebook

**Module:** Member 2 (Components A & B)  
**Core Mandate:** All numerical estimates, uncertainty distributions, and signals are derived strictly through ML and statistical analysis — never generated by free-form LLM hallucinations.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Mock Data Generator (Member 1 format)
from mock_data.generator import generate_synthetic_data

# 2. Component A: Market Signal Intelligence
from market_signals.pipeline import detect_all_signals

# 3. Component B: Forecasting Models & Baselines
from forecasting.baselines import SeasonalNaiveForecaster, MovingAverageForecaster
from forecasting.gb_quantile import GradientBoostedQuantileForecaster, FeatureBuilder
from forecasting.statistical import HierarchicalBayesPoolingForecaster
from forecasting.ablation import FeatureAblationStudy

# 4. Production Shared API for Member 3 (Optimizer)
from forecasting.api import forecast

print('All modules loaded successfully!')

## 1. Load Synthetic Retail Dataset
We load sales history, context (calendar, payday, macro, shocks), catalogue metadata, and customer reviews.

In [2]:
data = generate_synthetic_data(seed=42)
df_sales = data['sales']
df_context = data['context']
df_products = data['products']
df_reviews = data['reviews']

print(f'Sales records: {len(df_sales)}')
print(f'Context records: {len(df_context)}')
print(f'Product catalogue: {len(df_products)}')
df_products.head()

## 2. Component A: Market Signal Intelligence
Extract statistical sales velocity change points (CUSUM) and classify customer reviews into structured signals.

In [3]:
signals_df = detect_all_signals(df_sales, df_reviews, store_id='STORE_001')
print(f'Extracted {len(signals_df)} structured market signals conforming to contract:')
signals_df.head(10)

## 3. Component B: 4-Tier Feature Ablation Study
We evaluate 4 feature tiers out-of-sample over rolling origin windows to verify the gating rule:
- Tier 1: Sales only
- Tier 2: + Calendar & Payday
- Tier 3: + Economic context
- Tier 4: + Shocks & Signals

In [4]:
study = FeatureAblationStudy(horizon_days=14, n_splits=2)
ablation_results = study.run(df_sales, df_context, signals_df)
ablation_results

## 4. Production API: `forecast()` for Member 3
The optimizer calls `forecast(store_id, product_ids, horizon_days)` to get probabilistic demand distributions across products.

In [5]:
# Forecast for STORE_001 over a 14-day replenishment horizon
fc_df = forecast(
    store_id='STORE_001',
    product_ids=['PROD_MILK_1L', 'PROD_COOKING_OIL_1L', 'PROD_SPECIALTY_TEA', 'PROD_NEW_ENERGY_DRINK'],
    horizon_days=14
)
fc_df[['store_id', 'product_id', 'horizon_days', 'forecast_date', 'p10', 'p50', 'p90', 'shock_active', 'widening_factor', 'model_name']]

## 5. Visualizing Probabilistic Demand Distributions (Quantile Grid)

In [6]:
milk_row = fc_df[fc_df['product_id'] == 'PROD_MILK_1L'].iloc[0]
q_dict = milk_row['quantiles']
percentiles = [int(k.replace('q', '')) for k in sorted(q_dict.keys())]
quantiles_vals = [q_dict[k] for k in sorted(q_dict.keys())]

plt.figure(figsize=(10, 5))
plt.plot(percentiles, quantiles_vals, marker='o', color='#2563eb', linewidth=2.5, label='Demand Quantile Curve')
plt.axhline(milk_row['p50'], color='#10b981', linestyle='--', label=f'Median p50 = {milk_row["p50"]}')
plt.fill_between(percentiles, min(quantiles_vals), quantiles_vals, alpha=0.15, color='#3b82f6')
plt.title('Probabilistic Demand Forecast Curve (PROD_MILK_1L, 14d Horizon)', fontsize=13, fontweight='bold')
plt.xlabel('Quantile Percentile (q%)', fontsize=11)
plt.ylabel('Cumulative Units Demanded', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True)
plt.show()